# EcoRewind 2.0 — Phase 5 GPU runs (Colab **T4 GPU + High-RAM**)

Every run is resumable: if Colab disconnects, re-run cell 1 and the same run cell; finished folds are skipped.
Run the sections in any order. Approximate cost at ~1.6 CCU/h:

| Section | Run id | T4 hours | CCU |
|---|---|---|---|
| A. Ablations (EcoTransformer, Everglades B, 7 variants) | phase5_ablations | ~14 | ~22 |
| B. Cross-site transfer (SimVP + EcoTransformer, both sites, Track B) | phase5_transfer | ~9 | ~15 |
| C. Resolution: deep models on Mississippi Track B | phase5_resolution | ~8 | ~13 |
| D. Neighbour-informed SimVP (exploratory, A5) | phase5_neighbour | ~12 | ~19 |

### 1. Setup (every session)

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
TOKEN = userdata.get('GITHUB_TOKEN')
REPO = '/content/ecorewind2'
if os.path.isdir(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://{TOKEN}@github.com/DAISHINKAN7/EcoRewind2.git {REPO}
%cd {REPO}
!git log --oneline -1
!pip -q install -r requirements.txt
D = '/content/drive/MyDrive/EcoRewind2'
X = '/content/drive/MyDrive/EcoRewind2_exports'
!python scripts/cubes_cache.py --drive {D}/cubes --local /content/cubes --exports-drive {X} 2>&1 | tail -4
!mkdir -p /content/ppost && cp -n {X}/*PPOST* /content/ppost/ 2>/dev/null; ls /content/ppost | wc -l
RUN = lambda run_id, site, track, models, extra='': get_ipython().system(
    f"python -u scripts/run_baselines.py --cubes /content/cubes --phase1 {D}/phase1 --out {D}/runs --run-id {run_id} "
    f"--site {site} --track {track} --models {models} --table-cache /content/tables --ppost-exports /content/ppost {extra} "
    f"2>&1 | tee -a /content/{run_id}.log")

### A. Ablations (amendment A5)

In [ ]:
RUN('phase5_ablations', 'everglades', 'B',
    'ecotransformer_off_s0 ecotransformer_concat_s0 ecotransformer__smooth_s0 ecotransformer__nostatic_s0 '
    'ecotransformer__nomask_s0 ecotransformer__tin4_s0 ecotransformer__tin12_s0')

### B. Cross-site transfer

In [ ]:
RUN('phase5_transfer', 'everglades', 'B', 'simvp__xfer_s0 ecotransformer__xfer_s0')    # trained on Mississippi B
RUN('phase5_transfer', 'mississippi', 'B', 'simvp__xfer_s0 ecotransformer__xfer_s0')   # trained on Everglades B

### C. Resolution comparison (Mississippi Track B)

In [ ]:
RUN('phase5_resolution', 'mississippi', 'B', 'simvp_s0 convlstm_s0 ecotransformer_s0')

### D. Neighbour-informed SimVP (exploratory; never in the headline verdict)

In [ ]:
RUN('phase5_neighbour', 'everglades', 'B', 'simvp__nbr_s0 simvp__nbr_s1')
RUN('phase5_neighbour', 'mississippi', 'A', 'simvp__nbr_s0 simvp__nbr_s1')

When a section finishes: **Runtime → Disconnect and delete runtime**, and note the CCU balance.